# GROUP BY and HAVING

## Overview
`GROUP BY` collapses multiple rows that share the same value(s) in the grouping columns into a single summary row. Aggregate functions (COUNT, SUM, AVG, etc.) then compute a single value per group.

**Logical execution order with GROUP BY:**
```
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

This order determines what you can reference where:
- `WHERE` filters *individual rows* before grouping — cannot use aggregate functions
- `HAVING` filters *groups* after aggregation — can use aggregate functions
- `SELECT` aliases are not yet available in HAVING (PostgreSQL allows this as an extension; SQLite does not)

**Rules:**
1. Every column in SELECT that is not inside an aggregate function **must appear in GROUP BY**
2. You can GROUP BY expressions, not just column names: `GROUP BY STRFTIME('%Y-%m', txn_date)`
3. `GROUP BY` implicitly sorts in most databases — but don't rely on this; always add `ORDER BY`

**PostgreSQL note:** PostgreSQL allows referencing SELECT aliases in `ORDER BY` but not in `HAVING`. SQLite is stricter in some versions — use the full expression in HAVING to be safe.

---

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE accounts (
    account_id   INTEGER PRIMARY KEY,
    customer_id  INTEGER,
    account_type TEXT,      -- Chequing, Savings, Investment, Loan
    province     TEXT,
    opened_date  TEXT,
    status       TEXT       -- Active, Closed, Suspended
);
CREATE TABLE customers (
    customer_id  INTEGER PRIMARY KEY,
    first_name   TEXT,
    last_name    TEXT,
    segment      TEXT,      -- Retail, SME, Wealth
    province     TEXT
);
CREATE TABLE transactions (
    txn_id       INTEGER PRIMARY KEY,
    account_id   INTEGER,
    txn_date     TEXT,
    txn_type     TEXT,      -- Deposit, Withdrawal, Transfer, Fee, Interest
    amount       REAL,      -- positive = credit, negative = debit
    category     TEXT,      -- Payroll, Rent, Groceries, Utilities, etc.
    flagged      INTEGER    -- 1 = flagged for review
);
CREATE TABLE loans (
    loan_id      INTEGER PRIMARY KEY,
    customer_id  INTEGER,
    loan_type    TEXT,      -- Mortgage, Auto, Personal, HELOC
    principal    REAL,
    rate_pct     REAL,
    term_months  INTEGER,
    issued_date  TEXT,
    status       TEXT       -- Current, Delinquent, Paid Off, Default
);

INSERT INTO customers VALUES
  (1,'Aroha','Ngata','Retail','NB'),
  (2,'Liam','Chen','SME','BC'),
  (3,'Fatima','Al-Rashid','Wealth','ON'),
  (4,'James','MacLeod','Retail','NB'),
  (5,'Sofia','Petrov','SME','BC'),
  (6,'Noah','Williams','Retail','AB'),
  (7,'Mei','Zhang','Wealth','ON'),
  (8,'Ethan','Tremblay','Retail','QC'),
  (9,'Priya','Nair','SME','BC'),
  (10,'Marcus','Okafor','Retail','ON');

INSERT INTO accounts VALUES
  (101,1,'Chequing','NB','2019-03-15','Active'),
  (102,1,'Savings','NB','2019-03-15','Active'),
  (103,2,'Chequing','BC','2020-07-01','Active'),
  (104,2,'Investment','BC','2021-01-10','Active'),
  (105,3,'Chequing','ON','2018-05-20','Active'),
  (106,3,'Investment','ON','2018-05-20','Active'),
  (107,3,'Savings','ON','2022-11-01','Active'),
  (108,4,'Chequing','NB','2015-09-30','Active'),
  (109,4,'Loan','NB','2020-04-01','Closed'),
  (110,5,'Chequing','BC','2021-06-15','Active'),
  (111,5,'Savings','BC','2021-06-15','Suspended'),
  (112,6,'Chequing','AB','2017-11-22','Active'),
  (113,7,'Investment','ON','2016-03-08','Active'),
  (114,7,'Chequing','ON','2016-03-08','Active'),
  (115,8,'Chequing','QC','2023-01-05','Active'),
  (116,9,'Chequing','BC','2022-08-19','Active'),
  (117,9,'Investment','BC','2022-08-19','Active'),
  (118,10,'Chequing','ON','2020-12-01','Active'),
  (119,10,'Savings','ON','2020-12-01','Active');

INSERT INTO transactions VALUES
  (1001,101,'2023-01-05','Deposit',  4200.00,'Payroll',0),
  (1002,101,'2023-01-12','Withdrawal',-850.00,'Rent',0),
  (1003,101,'2023-01-20','Withdrawal',-120.00,'Groceries',0),
  (1004,101,'2023-02-05','Deposit',  4200.00,'Payroll',0),
  (1005,101,'2023-02-18','Withdrawal',-980.00,'Rent',0),
  (1006,102,'2023-01-15','Deposit',   500.00,'Transfer',0),
  (1007,102,'2023-03-10','Interest',   12.50,'Interest',0),
  (1008,103,'2023-01-08','Deposit', 15000.00,'Payroll',0),
  (1009,103,'2023-01-25','Withdrawal',-3200.00,'Rent',0),
  (1010,103,'2023-02-08','Deposit', 15000.00,'Payroll',0),
  (1011,104,'2023-01-31','Deposit',  5000.00,'Transfer',0),
  (1012,105,'2023-01-10','Deposit', 22000.00,'Payroll',0),
  (1013,105,'2023-01-28','Withdrawal',-8500.00,'Rent',0),
  (1014,105,'2023-02-10','Deposit', 22000.00,'Payroll',0),
  (1015,106,'2023-02-01','Deposit', 50000.00,'Investment',0),
  (1016,108,'2023-01-06','Deposit',  3100.00,'Payroll',0),
  (1017,108,'2023-01-19','Withdrawal',-700.00,'Rent',0),
  (1018,108,'2023-02-06','Deposit',  3100.00,'Payroll',0),
  (1019,110,'2023-01-15','Deposit',  8500.00,'Payroll',0),
  (1020,110,'2023-02-01','Withdrawal',-2100.00,'Rent',0),
  (1021,112,'2023-01-20','Deposit',  3800.00,'Payroll',0),
  (1022,112,'2023-02-10','Fee',       -25.00,'Fee',1),
  (1023,113,'2023-01-05','Deposit', 75000.00,'Investment',0),
  (1024,114,'2023-01-12','Deposit', 12000.00,'Payroll',0),
  (1025,114,'2023-02-12','Deposit', 12000.00,'Payroll',0),
  (1026,115,'2023-01-10','Deposit',  2800.00,'Payroll',0),
  (1027,115,'2023-01-28','Withdrawal',-650.00,'Groceries',0),
  (1028,116,'2023-01-18','Deposit',  9200.00,'Payroll',0),
  (1029,117,'2023-02-05','Deposit', 10000.00,'Investment',0),
  (1030,118,'2023-01-22','Deposit',  4500.00,'Payroll',0),
  (1031,118,'2023-02-15','Withdrawal',-1100.00,'Utilities',0),
  (1032,119,'2023-02-20','Interest',   18.75,'Interest',0),
  (1033,101,'2023-03-05','Deposit',  4200.00,'Payroll',NULL),
  (1034,103,'2023-03-08','Deposit', 15000.00,'Payroll',0),
  (1035,112,'2023-03-15','Withdrawal',-450.00,'Groceries',1);

INSERT INTO loans VALUES
  (201,1,'Personal',  15000.00, 7.5, 36,'2021-06-01','Current'),
  (202,2,'Mortgage', 480000.00, 4.2,300,'2020-07-15','Current'),
  (203,3,'HELOC',    100000.00, 6.1, 60,'2019-02-28','Current'),
  (204,4,'Auto',      28000.00, 5.9, 60,'2020-04-01','Paid Off'),
  (205,5,'Mortgage', 390000.00, 4.8,300,'2021-06-20','Current'),
  (206,6,'Personal',   8000.00, 9.2, 24,'2022-03-10','Delinquent'),
  (207,7,'Mortgage', 750000.00, 3.9,300,'2018-09-01','Current'),
  (208,8,'Auto',      22000.00, 6.5, 48,'2023-01-15','Current'),
  (209,9,'Personal',  12000.00, 8.1, 36,'2022-08-25','Current'),
  (210,10,'Auto',     35000.00, 5.4, 60,'2021-03-01','Current'),
  (211,3,'Personal',  25000.00, 6.8, 48,'2020-11-15','Paid Off'),
  (212,7,'HELOC',    200000.00, 5.5, 60,'2022-06-01','Current');
""")
conn.commit()

print('Finance schema ready. Table row counts:')
for t in ['customers','accounts','transactions','loans']:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}: {n} rows')


Finance schema ready. Table row counts:
  customers: 10 rows
  accounts: 19 rows
  transactions: 35 rows
  loans: 12 rows


---
## Basic GROUP BY — summarising by category

In [2]:
# Transaction count and total by type
print('=== Transaction summary by type ===')
q = """
SELECT  txn_type,
        COUNT(*)                    AS txn_count,
        ROUND(SUM(amount), 2)       AS total_amount,
        ROUND(AVG(amount), 2)       AS avg_amount,
        ROUND(MIN(amount), 2)       AS min_amount,
        ROUND(MAX(amount), 2)       AS max_amount
FROM    transactions
GROUP BY txn_type
ORDER BY total_amount DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Account count by type and status
print()
print('=== Account count by type and status ===')
q2 = """
SELECT  account_type,
        status,
        COUNT(*)  AS accounts
FROM    accounts
GROUP BY account_type, status
ORDER BY account_type, status
"""
print(pd.read_sql(q2, conn).to_string(index=False))


=== Transaction summary by type ===
  txn_type  txn_count  total_amount  avg_amount  min_amount  max_amount
   Deposit         22     301100.00    13686.36       500.0    75000.00
  Interest          2         31.25       15.63        12.5       18.75
       Fee          1        -25.00      -25.00       -25.0      -25.00
Withdrawal         10     -18650.00    -1865.00     -8500.0     -120.00

=== Account count by type and status ===
account_type    status  accounts
    Chequing    Active        10
  Investment    Active         4
        Loan    Closed         1
     Savings    Active         3
     Savings Suspended         1


---
## WHERE vs HAVING — filtering before and after aggregation

In [3]:
# WHERE filters rows before grouping; HAVING filters groups after aggregation
print('=== WHERE (pre-aggregation) vs HAVING (post-aggregation) ===')

# WHERE: only include non-fee transactions before grouping
q_where = """
SELECT  txn_type,
        COUNT(*)               AS txn_count,
        ROUND(SUM(amount), 2)  AS total_amount
FROM    transactions
WHERE   txn_type <> 'Fee'      -- excludes Fee rows before any grouping
GROUP BY txn_type
ORDER BY total_amount DESC
"""
print('After WHERE txn_type <> Fee:')
print(pd.read_sql(q_where, conn).to_string(index=False))

# HAVING: keep only groups with more than 2 transactions
print()
q_having = """
SELECT  account_id,
        COUNT(*)               AS txn_count,
        ROUND(SUM(amount), 2)  AS net_balance_change
FROM    transactions
GROUP BY account_id
HAVING  COUNT(*) > 2           -- applied after grouping; cannot use alias here
ORDER BY txn_count DESC
"""
print('Groups with more than 2 transactions (HAVING COUNT(*) > 2):')
print(pd.read_sql(q_having, conn).to_string(index=False))

# Combining WHERE and HAVING
print()
q_both = """
SELECT  account_id,
        COUNT(*)               AS deposit_count,
        ROUND(SUM(amount), 2)  AS total_deposits
FROM    transactions
WHERE   txn_type = 'Deposit'   -- pre-filter: deposits only
GROUP BY account_id
HAVING  SUM(amount) > 10000    -- post-filter: groups with large total deposits
ORDER BY total_deposits DESC
"""
print('Accounts with > $10k in deposits (WHERE for type, HAVING for total):')
print(pd.read_sql(q_both, conn).to_string(index=False))


=== WHERE (pre-aggregation) vs HAVING (post-aggregation) ===
After WHERE txn_type <> Fee:
  txn_type  txn_count  total_amount
   Deposit         22     301100.00
  Interest          2         31.25
Withdrawal         10     -18650.00

Groups with more than 2 transactions (HAVING COUNT(*) > 2):
 account_id  txn_count  net_balance_change
        101          6             10650.0
        103          4             41800.0
        112          3              3325.0
        108          3              5500.0
        105          3             35500.0

Accounts with > $10k in deposits (WHERE for type, HAVING for total):
 account_id  deposit_count  total_deposits
        113              1         75000.0
        106              1         50000.0
        103              3         45000.0
        105              2         44000.0
        114              2         24000.0
        101              3         12600.0


---
## GROUP BY on expressions

In [4]:
# Group by date part — monthly transaction summary
print('=== Monthly transaction volume ===')
q = """
SELECT  STRFTIME('%Y-%m', txn_date)  AS month,
        COUNT(*)                      AS transactions,
        ROUND(SUM(CASE WHEN amount > 0 THEN amount ELSE 0 END), 2)  AS total_credits,
        ROUND(SUM(CASE WHEN amount < 0 THEN amount ELSE 0 END), 2)  AS total_debits
FROM    transactions
GROUP BY STRFTIME('%Y-%m', txn_date)
ORDER BY month
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Group by customer segment via a join
print()
print('=== Loan portfolio by customer segment ===')
q2 = """
SELECT  c.segment,
        COUNT(l.loan_id)             AS loans,
        ROUND(SUM(l.principal), 2)   AS total_principal,
        ROUND(AVG(l.rate_pct), 2)    AS avg_rate_pct,
        COUNT(CASE WHEN l.status = 'Delinquent' THEN 1 END) AS delinquent
FROM    loans AS l
INNER JOIN customers AS c ON l.customer_id = c.customer_id
GROUP BY c.segment
ORDER BY total_principal DESC
"""
print(pd.read_sql(q2, conn).to_string(index=False))


=== Monthly transaction volume ===
  month  transactions  total_credits  total_debits
2023-01            19      165600.00      -14020.0
2023-02            12      116318.75       -4205.0
2023-03             4       19212.50        -450.0

=== Loan portfolio by customer segment ===
segment  loans  total_principal  avg_rate_pct  delinquent
 Wealth      4        1075000.0          5.58           0
    SME      3         882000.0          5.70           0
 Retail      5         108000.0          6.90           1


---
## HAVING with multiple conditions and aggregates

In [5]:
# Accounts that are both high-volume AND have flagged transactions
print('=== High-activity accounts with at least one flagged transaction ===')
q = """
SELECT  a.account_id,
        a.account_type,
        a.province,
        COUNT(t.txn_id)                AS txn_count,
        ROUND(SUM(t.amount), 2)        AS net_amount,
        SUM(COALESCE(t.flagged, 0))    AS flagged_count
FROM    accounts     AS a
INNER JOIN transactions AS t ON a.account_id = t.account_id
GROUP BY a.account_id, a.account_type, a.province
HAVING  COUNT(t.txn_id)              >= 3       -- active accounts
  AND   SUM(COALESCE(t.flagged, 0))  >= 1       -- at least one flagged txn
ORDER BY flagged_count DESC, txn_count DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Province-level loan risk summary
print()
print('=== Provinces with average loan rate above 6% ===')
q2 = """
SELECT  c.province,
        COUNT(l.loan_id)            AS loans,
        ROUND(AVG(l.rate_pct), 2)   AS avg_rate,
        ROUND(SUM(l.principal), 2)  AS total_exposure
FROM    loans      AS l
INNER JOIN customers AS c ON l.customer_id = c.customer_id
GROUP BY c.province
HAVING  AVG(l.rate_pct) > 6.0
ORDER BY avg_rate DESC
"""
print(pd.read_sql(q2, conn).to_string(index=False))

conn.close()


=== High-activity accounts with at least one flagged transaction ===
 account_id account_type province  txn_count  net_amount  flagged_count
        112     Chequing       AB          3      3325.0              2

=== Provinces with average loan rate above 6% ===
province  loans  avg_rate  total_exposure
      AB      1       9.2          8000.0
      NB      2       6.7         43000.0
      QC      1       6.5         22000.0


---
## Common Pitfalls

**1. Selecting a non-aggregated column that is not in GROUP BY**
`SELECT account_id, txn_type, COUNT(*) FROM transactions GROUP BY account_id` — `txn_type` is not in GROUP BY and not aggregated. PostgreSQL raises an error immediately. SQLite picks an arbitrary value for `txn_type` from the group. Always either aggregate or group by every non-aggregated column in SELECT.

**2. Using WHERE to filter on an aggregate function**
`WHERE COUNT(*) > 2` raises a syntax error — aggregate functions are not allowed in WHERE. Move aggregate filters to HAVING. WHERE runs before GROUP BY; HAVING runs after.

**3. Using a SELECT alias in HAVING**
`SELECT SUM(amount) AS total HAVING total > 1000` works in some databases (MySQL, BigQuery) but fails in PostgreSQL and SQLite. Write the full expression in HAVING: `HAVING SUM(amount) > 1000`.

**4. GROUP BY on a nullable column treats all NULLs as one group**
If `category` has three NULL rows, they are grouped together as a single NULL group. `COALESCE(category, 'Unknown')` in the GROUP BY creates a named group instead of a NULL group.

**5. Relying on GROUP BY for implicit sort order**
Many databases sort results by the GROUP BY columns as a side effect, but this is not guaranteed. PostgreSQL and SQLite may not. Always add an explicit `ORDER BY` when the sequence of grouped results matters.

**6. COUNT(*) vs COUNT(col) in grouped results**
`COUNT(*)` counts all rows in the group including those with NULL values in any column. `COUNT(flagged)` counts only rows where `flagged IS NOT NULL`. This distinction is critical when computing rates or checking for completeness within groups.


---
*sql_methods_library - Samantha McGarrigle*